In [ ]:
# ==============================================================================
# BANCADA DE PROVOCAÇÃO: TESTE DE CAUSALIDADE DE GRANGER EM NÍVEL & GRÁFICOS
# Comparação do EPU (Baker, Bloom & Davis) vs. VIX Tupiniquim (PCA e LASSO)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

print("--- [BANCADA DE PROVOCAÇÃO]: ANÁLISE COMPLETA EM NÍVEL (VIX PCA & LASSO vs EPU) ---")

# ==============================================================================
# 1. CARREGAMENTO E ALINHAMENTO DAS SÉRIES HISTÓRICAS
# ==============================================================================
# Carrega a série do EPU do arquivo do Baker, Bloom & Davis
df_epu = pd.read_excel('epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index')
df_epu['Data'] = pd.to_datetime(df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01')
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

# Carrega os dois VIX (PCA e LASSO) gerados no pipeline principal
df_vix = pd.read_excel('tabela_vix_tupiniquim_pca.xlsx') 
df_vix['Data'] = pd.to_datetime(df_vix['Data'])

# Purificação Estrutural (STL) do EPU
print("Aplicando Filtro Estrutural STL (period=13) na série do EPU...")
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

# Merge e alinhamento amostral
df_analise = pd.merge(df_vix, df_epu[['Data', 'EPU_SA']], on='Data', how='inner').sort_values('Data').reset_index(drop=True)
print(f"[ALINHAMENTO]: Amostra comum final com {len(df_analise)} meses sincronizados.")

# Padronização de todas as séries para Base Média = 100 na amostra
df_analise['VIX_PCA_100'] = (df_analise['VIX_PCA'] / df_analise['VIX_PCA'].mean()) * 100
df_analise['VIX_LASSO_100'] = (df_analise['VIX_LASSO'] / df_analise['VIX_LASSO'].mean()) * 100
df_analise['EPU_Base100'] = (df_analise['EPU_SA'] / df_analise['EPU_SA'].mean()) * 100

# ==============================================================================
# 2. VERIFICAÇÃO DE ESTACIONARIEDADE (TESTE ADF EM NÍVEL)
# ==============================================================================
print("\n--- TESTE DE ESTACIONARIEDADE EM NÍVEL (ADF) ---")
p_vix_pca = adfuller(df_analise['VIX_PCA_100'])[1]
p_vix_lasso = adfuller(df_analise['VIX_LASSO_100'])[1]
p_epu = adfuller(df_analise['EPU_Base100'])[1]

print(f"-> P-valor VIX Tupiniquim (PCA - Nível):   {p_vix_pca:.4f} -> {'Estacionário I(0)' if p_vix_pca < 0.05 else 'Não Estacionário'}")
print(f"-> P-valor VIX Tupiniquim (LASSO - Nível): {p_vix_lasso:.4f} -> {'Estacionário I(0)' if p_vix_lasso < 0.05 else 'Não Estacionário'}")
print(f"-> P-valor EPU_SA (Nível):                 {p_epu:.4f} -> {'Estacionário I(0)' if p_epu < 0.05 else 'Não Estacionário'}")

# ==============================================================================
# 3. TESTE DE CAUSALIDADE DE GRANGER EM NÍVEL (Lags 1 a 3)
# ==============================================================================
max_lags = 3
print(f"\n--- EXECUTANDO TESTE DE CAUSALIDADE DE GRANGER EM NÍVEL (Lags 1 a {max_lags}) ---")

def extrair_granger(data_df, var_y, var_x, rotulo_h0):
    res = grangercausalitytests(data_df[[var_y, var_x]], maxlag=max_lags, verbose=False)
    linhas = []
    for lag in range(1, max_lags + 1):
        f_stat = res[lag][0]['ssr_ftest'][0]
        p_val = res[lag][0]['ssr_ftest'][1]
        status = "REJEITA H0 (Causa!)" if p_val < 0.05 else "Não Rejeita H0"
        linhas.append({
            'Relação de Causalidade (H0)': rotulo_h0,
            'Lag': lag,
            'Estatística F': f_stat,
            'p-valor': p_val,
            'Conclusão (5% sig.)': status
        })
    return linhas

tabela_granger_nivel = []

# 3.1 VIX (PCA) vs EPU
tabela_granger_nivel.extend(extrair_granger(df_analise, 'EPU_Base100', 'VIX_PCA_100', 'VIX (PCA) ↛ EPU (VIX PCA não causa Notícias)'))
tabela_granger_nivel.extend(extrair_granger(df_analise, 'VIX_PCA_100', 'EPU_Base100', 'EPU ↛ VIX (PCA) (Notícias não causam VIX PCA)'))

# 3.2 VIX (LASSO) vs EPU
tabela_granger_nivel.extend(extrair_granger(df_analise, 'EPU_Base100', 'VIX_LASSO_100', 'VIX (LASSO) ↛ EPU (VIX LASSO não causa Notícias)'))
tabela_granger_nivel.extend(extrair_granger(df_analise, 'VIX_LASSO_100', 'EPU_Base100', 'EPU ↛ VIX (LASSO) (Notícias não causam VIX LASSO)'))

df_tabela6 = pd.DataFrame(tabela_granger_nivel)

print("\n" + "="*95)
print("     TABELA 6 OFICIAL: TESTE DE CAUSALIDADE DE GRANGER EM NÍVEL (VIX I vs. EPU BRASIL)")
print("="*95)
print(df_tabela6.to_string(index=False, formatters={
    'Estatística F': '{:.4f}'.format,
    'p-valor': '{:.4f}'.format
}))
print("="*95)
df_tabela6.to_excel('tabela_6_vix1_granger_nivel.xlsx', index=False)

# ==============================================================================
# 4. GERAÇÃO DOS GRÁFICOS SEPARADOS (EIXO DUPLO + VIA NEGATIVA: SEM LEGENDA)
# ==============================================================================

# --- 4.1. GRÁFICO 1: VIX (PCA) vs EPU (EIXO DUPLO) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

color1 = 'navy'
ax1.set_xlabel('Anos', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via PCA (Base Média = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_PCA_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Índice EPU Brasil (Base Média = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_pca_vs_epu_pt.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n[IMAGEM GERADA]: 'vix_pca_vs_epu_pt.png' salva com sucesso!")

# --- 4.2. GRÁFICO 2: VIX (LASSO) vs EPU (EIXO DUPLO) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

color1 = 'firebrick'
ax1.set_xlabel('Anos', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via LASSO (Base Média = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_LASSO_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Índice EPU Brasil (Base Média = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_lasso_vs_epu_pt.png', dpi=300, bbox_inches='tight')
plt.show()
print("[IMAGEM GERADA]: 'vix_lasso_vs_epu_pt.png' salva com sucesso!")

print("\n--- PROCESSAMENTO CONCLUÍDO COM SUCESSO! ---")